# 🚀 ONNX Image Upscaler — Complete Pipeline for Everyone

**Author:** [Jym Nils Caballero]
**Contact:** [https://github.com/jymnils2 / jymnnils@gmail.com]

## The Reality

Image super-resolution has come a long way. Models capable of upscaling images to stunning resolutions are published every day. But most users don't have a powerful GPU in their computer. This creates a gap: repositories full of extraordinary models that only people with dedicated hardware can use. No GPU? You're left out.

Sound familiar? It does to me too. That's why I built this.

## The Solution

This notebook runs in the cloud. Google Colab gives you a free T4 GPU, and this notebook does the rest. No installations needed, no special hardware required, no programming knowledge necessary. All you need is a browser.

- **Windows PC** → works.
- **Mac** → works.
- **Chromebook** → works.
- **Android phone** → works.
- **iPhone** → works.

If you have a browser, you have a GPU.

## What's Included

This isn't a basic notebook. It's a professional pipeline with everything you need:

| Section | What It Does |
|---|---|
| **Inference** | Upscales images with intelligent GPU tiling for any resolution. Downloads in PNG and JPG with embedded metadata. |
| **Conversion** | Converts PyTorch models to ONNX with one click. Automatic architecture detection via Spandrel. fp32/fp16 precision selection. |
| **Models** | Full model card with information pulled from OpenModelDB. Editable personal comments to organize your collection. Safe deletion. |
| **Drive** | Persistent storage on Google Drive. Your models, cards, and comments survive Colab session closures. Automatic sync between sessions. |

## Technical Features

- Tile processing with overlap to avoid artifacts at seams
- PyTorch → ONNX conversion with dynamic axes for full tiling compatibility
- OpenModelDB scraping to generate model cards with architecture, scale, dataset, license, and author info
- Embedded metadata in both ONNX files and output PNG/JPG files
- Overwrite protection with automatic renaming
- Professional web interface with 3 tabs (Gradio 6)

## Key Dependencies

Gradio 6 · ONNX Runtime GPU · PyTorch 2.11 · Spandrel · BeautifulSoup4 · Google Drive API

## Credits

- [ChaiNNer](https://github.com/chaiNNer-org/chaiNNer) — Inference pipeline inspiration
- [OpenModelDB](https://openmodeldb.info/) — Super-resolution model database
- [Spandrel](https://github.com/chaiNNer-org/spandrel) — Automatic architecture detection
- [ONNX Runtime](https://onnxruntime.ai/) — GPU-accelerated inference engine

## **Cell 1 — DEPENDENCIES**

> **Dependency installation with UV**
>
> This cell sets up the complete project environment. It uses UV, a high-speed package manager, to install all necessary libraries: Gradio 6 for the web interface, ONNX Runtime with GPU support for inference, PyTorch for model conversion, Spandrel for automatic architecture detection, BeautifulSoup4 for OpenModelDB scraping, and ONNX Script for export.
>
> All versions are pinned to guarantee compatibility and reproducibility. This cell is mandatory and must be executed before any other.

In [ ]:
# ==========================================
# CELDA 1: Instalación de dependencias con UV
# ==========================================

# 1. Instalar el gestor de paquetes uv
!pip install -q uv

# 2. Instalar dependencias del proyecto usando uv
# Versiones fijadas según entorno probado en Colab T4 (julio 2025)
!uv pip install --system \
    "torch==2.11.0" \
    "torchvision==0.26.0" \
    "gradio>=6.0.0,<7.0.0" \
    "onnxruntime-gpu>=1.20.0,<1.27.0" \
    "onnx==1.22.0" \
    "numpy>=2.0.0,<3.0.0" \
    "spandrel==0.4.2" \
    "spandrel_extra_arches" \
    "requests" \
    "onnxscript==0.7.1" \
    "beautifulsoup4" \
    "onnxconverter-common"

print("\n¡Entorno configurado correctamente con UV!")

Using Python 3.12.13 environment at: /usr
Resolved 91 packages in 163ms
Prepared 1 package in 16ms
Installed 1 package in 1ms
 + onnxconverter-common==1.16.0

¡Entorno configurado correctamente con UV!


## **Cell 2 — PERSISTENT STORAGE WITH GOOGLE DRIVE**

> **Persistent storage with Google Drive (Optional)**
>
> This cell connects Google Drive and redirects the local models directory (`models_uploaded`) so that it stores directly in Drive. This allows models to persist between Colab sessions: every converted ONNX, every companion JSON, and every personal comment is automatically synced to your Google Drive.
>
> If you don't run this cell, everything still works but models are lost when closing Colab. It's completely optional.

In [ ]:
# ==========================================
# CELDA 2: Almacenamiento persistente con Google Drive (Opcional)
# ==========================================
import os
import shutil
import json

DRIVE_MODELS_DIR = "/content/drive/MyDrive/modelos_ONNX"
LOCAL_MODELS_DIR = "modelos_subidos"

print("📁 Montando Google Drive...")
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive montado correctamente.")
except Exception as e:
    print(f"❌ Error al montar Drive: {e}")
    print("   Los modelos se guardarán localmente (se pierden al cerrar Colab).")
else:
    os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
    print(f"📂 Directorio en Drive: {DRIVE_MODELS_DIR}")

    # Si existe un directorio local con archivos, sincronizar con Drive
    if os.path.isdir(LOCAL_MODELS_DIR) and not os.path.islink(LOCAL_MODELS_DIR):
        archivos_locales = [
            f for f in os.listdir(LOCAL_MODELS_DIR)
            if os.path.isfile(os.path.join(LOCAL_MODELS_DIR, f))
        ]

        if archivos_locales:
            stats = {"nuevos": 0, "actualizados": 0, "existentes": 0}

            for archivo in archivos_locales:
                origen = os.path.join(LOCAL_MODELS_DIR, archivo)
                destino = os.path.join(DRIVE_MODELS_DIR, archivo)

                if not os.path.exists(destino):
                    shutil.copy2(origen, destino)
                    stats["nuevos"] += 1
                    print(f"   ✅ Nuevo: {archivo}")

                elif archivo.endswith(".json"):
                    try:
                        with open(destino, "r", encoding="utf-8") as f:
                            drive_data = json.load(f)
                        with open(origen, "r", encoding="utf-8") as f:
                            local_data = json.load(f)

                        comentarios_local = local_data.get("comentarios", "").strip()
                        comentarios_drive = drive_data.get("comentarios", "").strip()

                        if comentarios_local and comentarios_local != comentarios_drive:
                            drive_data["comentarios"] = comentarios_local
                            with open(destino, "w", encoding="utf-8") as f:
                                json.dump(drive_data, f, ensure_ascii=False, indent=2)
                            stats["actualizados"] += 1
                            print(f"   🔄 Actualizado (comentarios): {archivo}")
                        else:
                            stats["existentes"] += 1
                            print(f"   ⏭️ Idéntico: {archivo}")
                    except Exception:
                        stats["existentes"] += 1
                        print(f"   ⏭️ Sin cambios (error al leer): {archivo}")

                else:
                    tam_local = os.path.getsize(origen)
                    tam_drive = os.path.getsize(destino)
                    if tam_local != tam_drive:
                        stats["existentes"] += 1
                        print(f"   ⏭️ Diferente (Drive conservado): {archivo}")
                    else:
                        stats["existentes"] += 1
                        print(f"   ⏭️ Idéntico: {archivo}")

            print(f"\n📊 Sincronización completada:")
            print(f"   Nuevos copiados:      {stats['nuevos']}")
            print(f"   JSON actualizados:    {stats['actualizados']}")
            print(f"   Ya existían:          {stats['existentes']}")

            shutil.rmtree(LOCAL_MODELS_DIR)
            print("🗑️ Directorio local eliminado (contenido en Drive).")

    # Crear symlink
    if not os.path.exists(LOCAL_MODELS_DIR):
        os.symlink(DRIVE_MODELS_DIR, LOCAL_MODELS_DIR)
        print(f"🔗 Enlace creado: {LOCAL_MODELS_DIR} → {DRIVE_MODELS_DIR}")
    elif os.path.islink(LOCAL_MODELS_DIR):
        print(f"🔗 Enlace ya existe: {LOCAL_MODELS_DIR} → {os.readlink(LOCAL_MODELS_DIR)}")

    # Reporte final
    onnx_en_drive = [f for f in os.listdir(DRIVE_MODELS_DIR) if f.endswith('.onnx')]
    json_en_drive = [f for f in os.listdir(DRIVE_MODELS_DIR) if f.endswith('.json')]
    print(f"\n📊 Modelos en Drive: {len(onnx_en_drive)} ONNX, {len(json_en_drive)} JSON")
    for m in sorted(onnx_en_drive):
        print(f"   • {m}")

    print("\n✅ Listo. Los modelos se leen/escriben directamente en Google Drive.")

## **Cell 3 — INFERENCE CODE**

> **Backend: inference, conversion and model management**
>
> This cell contains all the project logic, divided into two blocks. The first implements image inference via ONNX models, with tile processing support, automatic scale detection, and result export with embedded metadata in PNG and JPG.
>
> The second block adds PyTorch → ONNX conversion functionality using Spandrel, OpenModelDB scraping with BeautifulSoup to generate complete model cards, personal comments management, and file overwrite protection.


In [ ]:
# ==========================================
# CELDA 3: Backend con Tiling, Metadatos, Doble Escalado y Conversión
# ==========================================
import os
import time
import math
import numpy as np
import onnxruntime as ort
from PIL import Image, PngImagePlugin
import gradio as gr

UPLOAD_DIR = "modelos_subidos"
OUTPUT_DIR = "resultados_descarga"
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

def obtener_metadatos_y_sugerencia(img_path):
    """Devuelve dimensiones, peso y una recomendación de Tile Size basada en la resolución."""
    if not img_path or not os.path.exists(img_path):
        return "N/A", "N/A", "Carga una imagen para ver la recomendación"

    with Image.open(img_path) as img:
        ancho, alto = img.size

    peso_bytes = os.path.getsize(img_path)
    peso_str = f"{peso_bytes / 1024:.2f} KB" if peso_bytes < 1024*1024 else f"{peso_bytes / (1024*1024):.2f} MB"

    lado_mayor = max(ancho, alto)
    if lado_mayor <= 1024:
        sugerencia = "💡 Desactivar Tiling o usar 1024px (Imagen pequeña)"
    elif lado_mayor <= 2500:
        sugerencia = "💡 Recomendado: 512px (Balance óptimo VRAM/Velocidad)"
    else:
        sugerencia = "💡 Recomendado: 256px (Imagen muy grande, previene OOM)"

    return f"{ancho} x {alto} px", peso_str, sugerencia

def listar_modelos():
    """Busca archivos .onnx en el directorio de trabajo."""
    archivos = [f for f in os.listdir(UPLOAD_DIR) if f.endswith('.onnx')]
    return archivos if archivos else ["No hay modelos subidos"]

def guardar_modelo(file_obj, sobrescribir=False):
    """Guarda un modelo .onnx subido por el usuario en disco, con protección contra sobreescritura."""
    if file_obj is None:
        return "No se seleccionó ningún archivo.", gr.Dropdown(choices=listar_modelos())
    nombre_archivo = os.path.basename(file_obj.name)
    destino = os.path.join(UPLOAD_DIR, nombre_archivo)
    destino, nombre_final, fue_renombrado = _resolver_conflicto_nombre(destino, sobrescribir)
    with open(file_obj.name, "rb") as f_in, open(destino, "wb") as f_out:
        f_out.write(f_in.read())
    if fue_renombrado:
        msg = f"Modelo guardado (renombrado para evitar conflicto): {nombre_final}"
    else:
        msg = f"Modelo guardado exitosamente: {nombre_final}"
    return msg, gr.Dropdown(choices=listar_modelos(), value=nombre_final)

def procesar_tile_onnx(session, tile_np, target_dtype, input_name, output_name):
    """Procesa un solo parche/tile a través del modelo ONNX."""
    img_input = np.transpose(tile_np, (2, 0, 1))[np.newaxis, ...].astype(target_dtype)
    raw_output = session.run([output_name], {input_name: img_input})[0]
    output_data = np.squeeze(raw_output, axis=0)
    output_data = np.transpose(output_data, (1, 2, 0))
    return output_data

def procesar_imagen_onnx(nombre_modelo, ruta_imagen_input, usar_tiling, tile_size, tile_pad, doble_escala=False, progress=gr.Progress()):
    if not ruta_imagen_input:
        yield None, None, None, "Error: No has seleccionado una imagen.", "", "", ""
        return
    if not nombre_modelo or nombre_modelo == "No hay modelos subidos":
        yield None, None, None, "Error: Selecciona un modelo ONNX válido.", "", "", ""
        return

    ruta_modelo = os.path.join(UPLOAD_DIR, nombre_modelo)
    dim_origen, peso_origen = obtener_metadatos_y_sugerencia(ruta_imagen_input)[:2]

    try:
        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
        session = ort.InferenceSession(ruta_modelo, providers=providers)

        input_tensor_info = session.get_inputs()[0]
        input_name = input_tensor_info.name
        expected_type = input_tensor_info.type
        output_name = session.get_outputs()[0].name

        target_dtype = np.float16 if "float16" in expected_type else np.float32
        img_pil = Image.open(ruta_imagen_input).convert("RGB")
        width, height = img_pil.size
        img_np = np.array(img_pil).astype(np.float32) / 255.0

        inicio = time.time()
        total_pases = 2 if doble_escala else 1

        for pase_actual in range(total_pases):
            pase_label = f"Pase {pase_actual + 1}/{total_pases}"

            if not usar_tiling:
                progress(0.5, desc=f"{pase_label}: Procesando imagen completa...")
                yield None, None, None, f"⏳ {pase_label}: Procesando imagen completa...", dim_origen, peso_origen, "Procesando..."

                output_tile = procesar_tile_onnx(session, img_np, target_dtype, input_name, output_name)
                output_img = np.clip(output_tile * 255.0, 0, 255).astype(np.uint8)
                img_out = Image.fromarray(output_img)

            else:
                tile_size, tile_pad = int(tile_size), int(tile_pad)

                # Solo detectar escala en el primer pase
                if pase_actual == 0:
                    sample_tile = img_np[:16, :16, :]
                    sample_out = procesar_tile_onnx(session, sample_tile, target_dtype, input_name, output_name)
                    scale = int(round(sample_out.shape[0] / 16))

                out_height, out_width = height * scale, width * scale
                output_img = np.zeros((out_height, out_width, 3), dtype=np.float32)

                num_tiles_x = math.ceil(width / tile_size)
                num_tiles_y = math.ceil(height / tile_size)
                total_tiles = num_tiles_x * num_tiles_y
                actual_tile = 0

                for y in range(num_tiles_y):
                    for x in range(num_tiles_x):
                        actual_tile += 1
                        status_progreso = f"{pase_label}: Tile {actual_tile}/{total_tiles}"
                        progress(actual_tile / total_tiles, desc=status_progreso)
                        yield None, None, None, f"⏳ {status_progreso}...", dim_origen, peso_origen, "Procesando..."

                        x_start, y_start = x * tile_size, y * tile_size
                        x_end, y_end = min(x_start + tile_size, width), min(y_start + tile_size, height)
                        x_start_pad, y_start_pad = max(x_start - tile_pad, 0), max(y_start - tile_pad, 0)
                        x_end_pad, y_end_pad = min(x_end + tile_pad, width), min(y_end + tile_pad, height)

                        tile = img_np[y_start_pad:y_end_pad, x_start_pad:x_end_pad, :]
                        tile_out = procesar_tile_onnx(session, tile, target_dtype, input_name, output_name)

                        pad_left = (x_start - x_start_pad) * scale
                        pad_top = (y_start - y_start_pad) * scale
                        pad_right = pad_left + (x_end - x_start) * scale
                        pad_bottom = pad_top + (y_end - y_start) * scale

                        tile_out_cropped = tile_out[pad_top:pad_bottom, pad_left:pad_right, :]

                        out_x_start, out_y_start = x_start * scale, y_start * scale
                        out_x_end = out_x_start + tile_out_cropped.shape[1]
                        out_y_end = out_y_start + tile_out_cropped.shape[0]

                        output_img[out_y_start:out_y_end, out_x_start:out_x_end, :] = tile_out_cropped

                output_img = np.clip(output_img * 255.0, 0, 255).astype(np.uint8)
                img_out = Image.fromarray(output_img)

            # Preparar para el siguiente pase (si hay)
            if pase_actual < total_pases - 1:
                img_np = np.array(img_out).astype(np.float32) / 255.0
                width, height = img_out.size

        # === PREPARAR PREVIEW LIGERA Y ARCHIVOS DE SALIDA ===
        img_preview = img_out.copy()
        img_preview.thumbnail((1200, 1200), Image.Resampling.LANCZOS)
        ruta_preview = os.path.join(OUTPUT_DIR, "preview_web.jpg")
        img_preview.save(ruta_preview, format="JPEG", quality=85)

        nombre_modelo_base = os.path.splitext(nombre_modelo)[0]
        nombre_orig_base = os.path.splitext(os.path.basename(ruta_imagen_input))[0]
        nombre_orig_truncado = nombre_orig_base[:12]
        nombre_salida_base = f"{nombre_orig_truncado}_{nombre_modelo_base}"

        ruta_png = os.path.join(OUTPUT_DIR, f"{nombre_salida_base}.png")
        ruta_jpg = os.path.join(OUTPUT_DIR, f"{nombre_salida_base}.jpg")

        # Metadatos PNG
        meta_png = PngImagePlugin.PngInfo()
        meta_png.add_text("Model", nombre_modelo)
        meta_png.add_text("Original_File", os.path.basename(ruta_imagen_input))
        meta_png.add_text("Software", "ChaiNNer ONNX Colab Pipeline")
        meta_png.add_text("Timestamp", time.strftime("%Y-%m-%d %H:%M:%S"))
        if doble_escala:
            meta_png.add_text("Upscale_Passes", "2")

        img_out.save(ruta_png, format="PNG", pnginfo=meta_png)

        # Metadatos EXIF en JPG
        exif_bytes = img_out.getexif()
        exif_bytes[0x010e] = f"Model: {nombre_modelo} | Original: {os.path.basename(ruta_imagen_input)}"
        exif_bytes[0x0131] = "ChaiNNer ONNX Colab Pipeline"
        img_out.save(ruta_jpg, format="JPEG", quality=90, optimize=True, exif=exif_bytes)

        tiempo_total = time.time() - inicio
        dim_salida, peso_salida = obtener_metadatos_y_sugerencia(ruta_png)[:2]
        status_final = f"✅ Procesado en {tiempo_total:.2f}s."
        if doble_escala:
            status_final += " (doble escalado)"

        yield (
            ruta_preview,
            ruta_png,
            ruta_jpg,
            status_final,
            dim_origen,
            peso_origen,
            f"{dim_salida} | {peso_salida}"
        )

    except Exception as e:
        yield None, None, None, f"❌ Error durante el procesamiento: {str(e)}", dim_origen, peso_origen, "Error"


# ═══════════════════════════════════════════════════════════════════════
# NUEVAS FUNCIONES — Conversión PyTorch → ONNX y Gestión de Modelos
# ═══════════════════════════════════════════════════════════════════════

import gc
import json
import re
import requests
import torch
import onnx
from bs4 import BeautifulSoup
from spandrel import ModelLoader, ImageModelDescriptor

if '_SPANDREL_EXTRA_INSTALLED' not in dir():
    try:
        import spandrel_extra_arches
        spandrel_extra_arches.install()
        _SPANDREL_EXTRA_OK = True
    except Exception:
        _SPANDREL_EXTRA_OK = False
        print("⚠️  spandrel_extra_arches no disponible. Solo arquitecturas base soportadas.")
    _SPANDREL_EXTRA_INSTALLED = True


# ── Utilidades internas ──────────────────────────────────────────────

def _get_json_path(model_filename):
    """Ruta del JSON companion para un modelo ONNX dado."""
    base = os.path.splitext(model_filename)[0]
    return os.path.join(UPLOAD_DIR, base + ".json")


def _resolver_conflicto_nombre(ruta_destino, sobrescribir=False):
    """
    Si el archivo ya existe:
      - sobrescribir=True  → devuelve la misma ruta
      - sobrescribir=False → genera nombre con sufijo _v2, _v3...
    Devuelve: (ruta_final, nombre_final, fue_renombrado)
    """
    if sobrescribir or not os.path.exists(ruta_destino):
        return ruta_destino, os.path.basename(ruta_destino), False

    directorio = os.path.dirname(ruta_destino)
    nombre_base, extension = os.path.splitext(os.path.basename(ruta_destino))
    nombre_limpio = re.sub(r'_v\d+$', '', nombre_base)

    version = 2
    while True:
        nuevo_nombre = f"{nombre_limpio}_v{version}{extension}"
        nueva_ruta = os.path.join(directorio, nuevo_nombre)
        if not os.path.exists(nueva_ruta):
            return nueva_ruta, nuevo_nombre, True
        version += 1


def _parse_openmodeldb_url(url):
    """Extrae el ID del modelo de una URL de OpenModelDB."""
    if not url or not isinstance(url, str):
        return None
    url = url.strip().rstrip("/")
    if "openmodeldb.info/models/" in url:
        return url.split("openmodeldb.info/models/")[-1]
    return None


def _scrape_openmodeldb_page(url):
    """
    Scrape de la página de OpenModelDB usando BeautifulSoup.
    Extrae todos los campos visibles: arquitectura, escala, tamaño,
    color mode, licencia, fecha, dataset, parámetros de entrenamiento,
    descripción, autor con URL de perfil e imágenes de muestra.
    Devuelve (data_dict, error_str).
    """
    model_id = _parse_openmodeldb_url(url)
    if not model_id:
        return None, "URL no válida. Formato esperado: https://openmodeldb.info/models/{id}"

    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                       "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }

    try:
        resp = requests.get(url.strip(), timeout=20, headers=headers)
        resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        return None, f"Error al acceder a OpenModelDB: {e}"

    soup = BeautifulSoup(resp.text, "html.parser")

    # ── Estructura base del resultado ──
    result = {
        "nombre": model_id,
        "descripcion": "",
        "url": url.strip(),
        "architecture": "",
        "scale": "",
        "size": "",
        "color_mode": "RGB",
        "license": "",
        "license_details": [],
        "date": "",
        "dataset": "",
        "dataset_size": "",
        "training_iterations": "",
        "training_epochs": "",
        "training_batch_size": "",
        "training_hr_size": "",
        "training_otf": "",
        "tags": [],
        "author": "",
        "author_url": "",
        "imagen_muestra": "",
    }

    # ── Nombre del modelo ──
    h1 = soup.find("h1")
    if h1:
        result["nombre"] = h1.get_text(strip=True)

    # ── Descripción (primer párrafo sustancial) ──
    for p in soup.find_all("p"):
        text = p.get_text(strip=True)
        if len(text) > 30:
            lower = text.lower()
            if "cookie" not in lower and "javascript" not in lower:
                result["descripcion"] = text[:500]
                break

    # ── Campos de especificaciones ──
    FIELD_MAP = {
        "architecture": "architecture",
        "scale": "scale",
        "size": "size",
        "color mode": "color_mode",
        "license": "license",
        "date": "date",
        "dataset": "dataset",
        "dataset size": "dataset_size",
        "training iterations": "training_iterations",
        "training epochs": "training_epochs",
        "training batch size": "training_batch_size",
        "training hr size": "training_hr_size",
        "training otf": "training_otf",
    }

    # Estrategia 1: Filas de tabla (tr > th/td)
    for row in soup.find_all("tr"):
        cells = row.find_all(["th", "td"])
        if len(cells) >= 2:
            label = cells[0].get_text(strip=True).lower().strip()
            if label in FIELD_MAP and not result[FIELD_MAP[label]]:
                result[FIELD_MAP[label]] = cells[1].get_text(strip=True)

    # Estrategia 2: Listas de definición (dt/dd)
    for dt in soup.find_all("dt"):
        label = dt.get_text(strip=True).lower().strip()
        if label in FIELD_MAP and not result[FIELD_MAP[label]]:
            dd = dt.find_next_sibling("dd")
            if dd:
                result[FIELD_MAP[label]] = dd.get_text(strip=True)

    # Estrategia 3: Buscar texto de etiqueta y valor en siguiente hermano
    found_keys = {v for k, v in FIELD_MAP.items() if result.get(v)}
    remaining = {k: v for k, v in FIELD_MAP.items() if v not in found_keys}

    if remaining:
        for label, key in remaining.items():
            if result[key]:
                continue
            for text_node in soup.find_all(
                string=lambda s: s and s.strip().lower() == label
            ):
                parent = text_node.find_parent()
                if not parent:
                    continue
                next_sib = parent.find_next_sibling()
                if next_sib:
                    val = next_sib.get_text(strip=True)
                    if val and val.lower() != label:
                        result[key] = val
                        break
                next_text = text_node.find_next_sibling()
                if next_text:
                    val = next_text.get_text(strip=True)
                    if val and val.lower() != label:
                        result[key] = val
                        break

    # ── Detalles de licencia (elementos de lista) ──
    for text_node in soup.find_all(
        string=lambda s: s and s.strip().lower() == "license"
    ):
        parent = text_node.find_parent()
        if not parent:
            continue
        container = parent.find_parent()
        if container:
            for li in container.find_all("li"):
                detail = li.get_text(strip=True)
                if detail and detail.lower() != "license":
                    result["license_details"].append(detail)
        break

    # ── Autor: buscar enlaces a perfil de usuario ──
    for a_tag in soup.find_all("a", href=True):
        href = a_tag["href"]
        if "/users/" in href:
            author_name = a_tag.get_text(strip=True)
            if author_name and len(author_name) > 1:
                result["author"] = author_name
                if href.startswith("http"):
                    result["author_url"] = href
                elif href.startswith("/"):
                    result["author_url"] = "https://openmodeldb.info" + href
                break

    # Si no encontró enlace directo, buscar como texto de etiqueta
    if not result["author"]:
        for text_node in soup.find_all(
            string=lambda s: s and s.strip().lower() == "author"
        ):
            parent = text_node.find_parent()
            if parent:
                next_el = parent.find_next_sibling()
                if next_el:
                    link = next_el.find("a", href=True)
                    if link:
                        result["author"] = link.get_text(strip=True)
                        href = link["href"]
                        if href.startswith("http"):
                            result["author_url"] = href
                        elif href.startswith("/"):
                            result["author_url"] = "https://openmodeldb.info" + href
                    else:
                        result["author"] = next_el.get_text(strip=True)
            break

    # ── Imágenes de muestra ──
    for img in soup.find_all("img"):
        src = img.get("src", "") or img.get("data-src", "")
        if not src:
            continue
        lower_src = src.lower()
        if any(skip in lower_src for skip in [
            "logo", "icon", "favicon", "avatar", "badge", "svg"
        ]):
            continue
        if src.startswith("//"):
            src = "https:" + src
        elif src.startswith("/"):
            src = "https://openmodeldb.info" + src
        if src.startswith("http"):
            result["imagen_muestra"] = src
            break

    return result, None


def _crear_json_base(archivo_original, arch_name, scale, in_channels, out_channels, precision):
    """Crea la estructura base de un JSON companion."""
    return {
        "nombre": os.path.splitext(archivo_original)[0],
        "descripcion": "",
        "url": "",
        "architecture": arch_name,
        "scale": f"{scale}x" if isinstance(scale, int) else str(scale),
        "size": "",
        "color_mode": "RGB",
        "license": "",
        "license_details": [],
        "date": "",
        "dataset": "",
        "dataset_size": "",
        "training_iterations": "",
        "training_epochs": "",
        "training_batch_size": "",
        "training_hr_size": "",
        "training_otf": "",
        "tags": [],
        "author": "",
        "author_url": "",
        "imagen_muestra": "",
        "comentarios": "",
        "fecha_conversion": time.strftime("%Y-%m-%d %H:%M:%S"),
        "archivo_original": archivo_original,
        "precision": precision,
        "channels_in": in_channels,
        "channels_out": out_channels,
    }


def _format_ficha(data):
    """Genera texto en formato Markdown legible con links clickeables."""

    url = data.get("url", "")
    url_md = f"[{url}]({url})" if url else "—"

    author = data.get("author", "")
    author_url = data.get("author_url", "")
    if author and author_url:
        author_md = f"[{author}]({author_url})"
    elif author:
        author_md = author
    else:
        author_md = "—"

    lic_details = data.get("license_details", [])
    if isinstance(lic_details, list) and lic_details:
        lic_extra = "\n".join(f"  - {d}" for d in lic_details)
    else:
        lic_extra = ""

    tags = data.get("tags", [])
    tags_str = ", ".join(tags) if isinstance(tags, list) and tags else "—"

    parts = [
        f"### 📌 {data.get('nombre', 'N/A')}",
        "",
        f"**Arquitectura:** {data.get('architecture', 'N/A') or 'N/A'}",
        f"**Escala:** {data.get('scale', 'N/A') or 'N/A'}",
        f"**Params:** {data.get('size', 'N/A') or 'N/A'}",
        f"**Color:** {data.get('color_mode', 'N/A') or 'N/A'}",
        f"**Canales:** {data.get('channels_in', '?')} → {data.get('channels_out', '?')}",
        f"**Precisión:** {data.get('precision', 'N/A') or 'N/A'}",
        "",
        "---",
        "",
        f"**Descripción:** {data.get('descripcion', '') or '—'}",
        f"**URL:** {url_md}",
        f"**Autor:** {author_md}",
        f"**Licencia:** {data.get('license', '') or '—'}",
    ]

    if lic_extra:
        parts.append(lic_extra)

    parts.extend([
        f"**Fecha:** {data.get('date', '') or '—'}",
        "",
        "---",
        "",
        f"**Dataset:** {data.get('dataset', '') or '—'}",
        f"**Dataset size:** {data.get('dataset_size', '') or '—'}",
        f"**Iteraciones:** {data.get('training_iterations', '') or '—'}",
        f"**Épocas:** {data.get('training_epochs', '') or '—'}",
        f"**Batch size:** {data.get('training_batch_size', '') or '—'}",
        f"**HR size:** {data.get('training_hr_size', '') or '—'}",
        f"**OTF:** {data.get('training_otf', '') or '—'}",
        "",
        "---",
        "",
        f"**Tags:** {tags_str}",
        f"**Archivo original:** {data.get('archivo_original', '—')}",
        f"**Conversión:** {data.get('fecha_conversion', '—')}",
    ])

    return "\n".join(parts)


# ── Funciones públicas ───────────────────────────────────────────────

def convertir_pytorch_a_onnx(archivo_modelo, url_openmodeldb, precision, sobrescribir=False, progress=gr.Progress()):
    """
    Convierte un modelo PyTorch (.pth / .safetensors / .ckpt / .pt) a ONNX
    usando Spandrel para la detección de arquitectura.
    Genera: ONNX en UPLOAD_DIR + JSON companion con sufijo de precisión.
    Si ya existe un ONNX con el mismo nombre y sobrescribir=False, se renombra.
    Yield:  (status_text, info_text)
    """
    if not archivo_modelo:
        yield "❌ Error: No se seleccionó ningún archivo.", ""
        return

    filepath = archivo_modelo.name if hasattr(archivo_modelo, "name") else str(archivo_modelo)
    filename = os.path.basename(filepath)

    # ── 0. Limpiar VRAM de sesiones anteriores ───────────────────────
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ── 1. Cargar con Spandrel ───────────────────────────────────────
    progress(0.05, desc="Detectando arquitectura con Spandrel...")
    yield "🔍 Detectando arquitectura del modelo...", ""

    try:
        loader = ModelLoader()
        model_descriptor = loader.load_from_file(filepath)
    except Exception as e:
        yield (
            f"❌ Spandrel no pudo cargar el modelo.\n"
            f"Arquitectura no soportada o archivo corrupto.\n\nDetalle: {e}",
            ""
        )
        return

    if not isinstance(model_descriptor, ImageModelDescriptor):
        yield (
            f"❌ El archivo no es un modelo de imagen válido.\n"
            f"Tipo detectado: {type(model_descriptor).__name__}",
            ""
        )
        return

    model = model_descriptor.model
    model.eval()

    try:    arch_name = model_descriptor.architecture.name
    except: arch_name = str(getattr(model_descriptor, "architecture", "Desconocida"))
    try:    scale = model_descriptor.scale
    except: scale = 4
    try:    in_ch = model_descriptor.in_channels
    except: in_ch = 3
    try:    out_ch = model_descriptor.out_channels
    except: out_ch = in_ch

    # Liberar descriptor completo, ya no lo necesitamos
    del model_descriptor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    info_text = (
        f"✅ Modelo detectado:\n"
        f"{'━' * 35}\n"
        f"Arquitectura: {arch_name}\n"
        f"Escala: {scale}x\n"
        f"Canales entrada: {in_ch}\n"
        f"Canales salida: {out_ch}\n"
        f"Precisión destino: {precision}\n"
        f"{'━' * 35}"
    )
    yield f"✅ Detectado: {arch_name} ({scale}x). Preparando exportación...", info_text

    # ── 2. Preparar tensores ─────────────────────────────────────────
    progress(0.15, desc="Preparando tensores...")
    is_fp16 = precision == "fp16"
    try:
        dummy = torch.randn(1, in_ch, 64, 64).float()
    except Exception as e:
        yield f"❌ Error al preparar tensores: {e}", info_text
        return

    # ── 3. Resolver nombre del archivo de salida ─────────────────────
    model_base = os.path.splitext(filename)[0]
    precision_tag = "fp16" if is_fp16 else "fp32"
    onnx_nombre_inicial = f"{model_base}_{precision_tag}.onnx"
    onnx_path_inicial = os.path.join(UPLOAD_DIR, onnx_nombre_inicial)
    onnx_path, onnx_filename, fue_renombrado = _resolver_conflicto_nombre(onnx_path_inicial, sobrescribir)

    # Si se renombra, conservar comentarios del JSON anterior
    json_path_original = _get_json_path(onnx_nombre_inicial)
    comentarios_previos = ""
    if fue_renombrado and os.path.exists(json_path_original):
        try:
            with open(json_path_original, "r", encoding="utf-8") as f:
                json_anterior = json.load(f)
            comentarios_previos = json_anterior.get("comentarios", "")
        except Exception:
            pass

    if fue_renombrado:
        yield f"📝 Archivo renombrado a: {onnx_filename} (el original ya existe)", info_text

    # ── 4. Exportar a ONNX ───────────────────────────────────────────
    # ── 4. Exportar a ONNX (siempre fp32, conversión a fp16 después) ─
    progress(0.25, desc="Exportando a ONNX (siempre fp32)...")
    yield "⚙️ Exportando a ONNX...", info_text

    try:
        with torch.no_grad():
            torch.onnx.export(
                model,
                dummy,
                onnx_path,
                opset_version=20,
                input_names=["input"],
                output_names=["output"],
                dynamic_axes={
                    "input":  {0: "batch", 2: "height", 3: "width"},
                    "output": {0: "batch", 2: "height", 3: "width"},
                },
                do_constant_folding=True,
                export_params=True,
                dynamo=False,
            )
    except Exception as e:
        yield f"❌ Error durante la exportación ONNX: {e}", info_text
        return
    finally:
        del model, dummy
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── 4b. Conversión fp32 → fp16 si aplica ────────────────────────
    if is_fp16:
        progress(0.45, desc="Convirtiendo ONNX a fp16...")
        yield "⚙️ Convirtiendo a fp16...", info_text
        try:
            from onnxconverter_common import float16 as onnx_float16
            onnx_model = onnx.load(onnx_path)
            onnx_fp16 = onnx_float16.convert_float_to_float16_model(onnx_model)
            onnx.save(onnx_fp16, onnx_path)
            del onnx_model, onnx_fp16
        except Exception as e:
            yield f"⚠️ Conversión fp16 falló: {e}. El modelo se guardó en fp32.", info_text
            is_fp16 = False
            precision = "fp32"

    # ── 5. Metadatos embebidos en el ONNX ────────────────────────────
    progress(0.55, desc="Insertando metadatos en ONNX...")
    try:
        onnx_model = onnx.load(onnx_path)
        for key, value in [
            ("name",             model_base),
            ("architecture",     arch_name),
            ("scale",            f"{scale}x"),
            ("in_channels",      str(in_ch)),
            ("out_channels",     str(out_ch)),
            ("precision",        precision),
            ("original_file",    filename),
            ("conversion_date",  time.strftime("%Y-%m-%d %H:%M:%S")),
            ("software",         "ChaiNNer ONNX Colab Pipeline"),
        ]:
            entry = onnx_model.metadata_props.add()
            entry.key = key
            entry.value = str(value)
        onnx.save(onnx_model, onnx_path)
    except Exception as e:
        print(f"⚠️ No se pudieron añadir metadatos ONNX: {e}")

    # ── 6. Validación con ONNX Runtime ───────────────────────────────
    progress(0.65, desc="Validando ONNX...")
    yield "🔍 Validando modelo ONNX...", info_text

    try:
        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
        sess = ort.InferenceSession(onnx_path, providers=providers)
        dtype_test = np.float16 if is_fp16 else np.float32
        test_in = np.random.randn(1, in_ch, 64, 64).astype(dtype_test)
        test_out = sess.run(None, {"input": test_in})[0]
        val_msg = f"✅ ONNX válido. Salida de prueba: {test_out.shape}"
    except Exception as e:
        val_msg = f"⚠️ Validación falló: {e}. El archivo se guardó pero podría no funcionar."

    yield val_msg, info_text

    # ── 7. Construir JSON companion ──────────────────────────────────
    progress(0.8, desc="Generando ficha JSON...")
    json_data = _crear_json_base(os.path.basename(onnx_path), arch_name, scale, in_ch, out_ch, precision)

    # Restaurar comentarios previos si hubo renombrado
    if comentarios_previos:
        json_data["comentarios"] = comentarios_previos

    # ── 8. Integrar URL y OpenModelDB (si aplica) ────────────────────
    omd_status = ""
    if url_openmodeldb and url_openmodeldb.strip():
        url_limpia = url_openmodeldb.strip()
        json_data["url"] = url_limpia

        # Solo scrapear si es OpenModelDB
        if "openmodeldb.info" in url_limpia:
            progress(0.85, desc="Scrapeando OpenModelDB...")
            yield "🌐 Consultando OpenModelDB...", info_text

            scraped_data, error = _scrape_openmodeldb_page(url_limpia)
            if scraped_data:
                for k, v in scraped_data.items():
                    if k in json_data and v and k not in (
                        "fecha_conversion", "archivo_original", "precision",
                        "channels_in", "channels_out", "comentarios", "url",
                    ):
                        json_data[k] = v
                omd_status = "✅ Información de OpenModelDB integrada."
            else:
                omd_status = f"⚠️ No se pudo obtener info de OpenModelDB: {error}"
        else:
            omd_status = "📌 URL de origen guardada (no es OpenModelDB, sin scraping automático)."

    # Guardar JSON
    json_path = _get_json_path(onnx_filename)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(json_data, f, ensure_ascii=False, indent=2)

    progress(1.0, desc="¡Completado!")

    final = f"✅ Conversión completada: {onnx_filename}\n{val_msg}"
    if fue_renombrado:
        final += f"\n📝 Nombre original ya existía, guardado como: {onnx_filename}"
    if omd_status:
        final += f"\n{omd_status}"
    final += f"\n📁 ONNX → {onnx_path}\n📄 JSON → {json_path}"
    yield final, info_text


def cargar_ficha_modelo(nombre_modelo):
    """Carga el JSON companion de un modelo y devuelve (ficha, json_raw, comentarios)."""
    vacio = ("*Selecciona un modelo para ver su información.*", "{}", "")
    if not nombre_modelo or nombre_modelo == "No hay modelos subidos":
        return vacio

    json_path = _get_json_path(nombre_modelo)

    if os.path.exists(json_path):
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return (
            _format_ficha(data),
            json.dumps(data, ensure_ascii=False, indent=2),
            data.get("comentarios", ""),
        )

    # Sin JSON: intentar leer metadatos embebidos en el ONNX
    onnx_path = os.path.join(UPLOAD_DIR, nombre_modelo)
    if os.path.exists(onnx_path):
        try:
            om = onnx.load(onnx_path)
            meta = {p.key: p.value for p in om.metadata_props}
            if meta:
                lines = ["### ⚠️ Sin ficha JSON", "", "**Metadatos embebidos en ONNX:**", ""]
                for k, v in meta.items():
                    lines.append(f"**{k}:** {v}")
                return "\n".join(lines), "{}", ""
        except Exception:
            pass

    return (
        "⚠️ Sin información. Usa el campo inferior para generar la ficha desde OpenModelDB.",
        "{}",
        "",
    )


def guardar_comentarios(nombre_modelo, comentarios):
    """Guarda el campo 'comentarios' en el JSON companion."""
    if not nombre_modelo or nombre_modelo == "No hay modelos subidos":
        return "Error: No hay modelo seleccionado."

    json_path = _get_json_path(nombre_modelo)

    if os.path.exists(json_path):
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = _crear_json_base(nombre_modelo, "", "", 3, 3, "")

    data["comentarios"] = comentarios

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    return f"✅ Comentarios guardados para {nombre_modelo}."


def generar_json_desde_url(nombre_modelo, url):
    """
    Genera (o actualiza) el JSON companion de un modelo ya en disco.
    Si se proporciona URL de OpenModelDB, scrapea la info.
    Si no hay URL, crea un JSON base con lo que se pueda leer del ONNX.
    Devuelve: (status, ficha_formateada, json_raw, comentarios)
    """
    if not nombre_modelo or nombre_modelo == "No hay modelos subidos":
        return "Error: No hay modelo seleccionado.", "*Selecciona un modelo*", "{}", ""

    json_path = _get_json_path(nombre_modelo)

    # Conservar comentarios existentes si los hay
    comentarios_previos = ""
    fecha_previa = ""
    if os.path.exists(json_path):
        try:
            with open(json_path, "r", encoding="utf-8") as f:
                existing = json.load(f)
            comentarios_previos = existing.get("comentarios", "")
            fecha_previa = existing.get("fecha_conversion", "")
        except Exception:
            pass

    # Intentar leer metadatos embebidos del ONNX como base
    arch_from_onnx, scale_from_onnx, in_ch_from_onnx, out_ch_from_onnx = "", "", 3, 3
    onnx_path = os.path.join(UPLOAD_DIR, nombre_modelo)
    if os.path.exists(onnx_path):
        try:
            om = onnx.load(onnx_path)
            meta = {p.key: p.value for p in om.metadata_props}
            arch_from_onnx = meta.get("architecture", "")
            scale_from_onnx = meta.get("scale", "")
            in_ch_from_onnx = int(meta.get("in_channels", 3))
            out_ch_from_onnx = int(meta.get("out_channels", 3))
        except Exception:
            pass

    # Crear JSON base con lo que tengamos
    json_data = _crear_json_base(
        nombre_modelo, arch_from_onnx, scale_from_onnx,
        in_ch_from_onnx, out_ch_from_onnx, ""
    )
    json_data["comentarios"] = comentarios_previos
    if fecha_previa:
        json_data["fecha_conversion"] = fecha_previa

    omd_status = ""

    # Si hay URL
    if url and url.strip():
        url_limpia = url.strip()
        json_data["url"] = url_limpia

        if "openmodeldb.info" in url_limpia:
            scraped_data, error = _scrape_openmodeldb_page(url_limpia)
            if scraped_data:
                for k, v in scraped_data.items():
                    if k in json_data and v and k not in (
                        "fecha_conversion", "archivo_original", "precision",
                        "channels_in", "channels_out", "comentarios", "url",
                    ):
                        json_data[k] = v
                omd_status = "✅ Ficha generada desde OpenModelDB."
            else:
                omd_status = f"⚠️ URL guardada pero no se pudo scrapear: {error}"
        else:
            omd_status = "✅ Ficha base creada con URL de origen."
    else:
        omd_status = "✅ Ficha base creada (sin URL). Puedes añadir info manualmente."

    # Guardar JSON
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(json_data, f, ensure_ascii=False, indent=2)

    ficha, raw, comentarios = cargar_ficha_modelo(nombre_modelo)
    return omd_status, ficha, raw, comentarios


def leer_info_basica_modelo(nombre_modelo):
    """
    Lee información básica de un modelo para el panel de Inferencia.
    Devuelve un string corto: arquitectura, escala, color, params.
    """
    if not nombre_modelo or nombre_modelo == "No hay modelos subidos":
        return ""

    json_path = _get_json_path(nombre_modelo)
    if os.path.exists(json_path):
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        parts = []
        if data.get("architecture"):
            parts.append(f"Arq: {data['architecture']}")
        if data.get("scale"):
            parts.append(f"Escala: {data['scale']}")
        if data.get("color_mode"):
            parts.append(f"Color: {data['color_mode']}")
        if data.get("size"):
            parts.append(f"Params: {data['size']}")
        return " | ".join(parts) if parts else "Sin metadatos"

    onnx_path = os.path.join(UPLOAD_DIR, nombre_modelo)
    if os.path.exists(onnx_path):
        try:
            om = onnx.load(onnx_path)
            meta = {p.key: p.value for p in om.metadata_props}
            parts = []
            if "architecture" in meta:
                parts.append(f"Arq: {meta['architecture']}")
            if "scale" in meta:
                parts.append(f"Escala: {meta['scale']}")
            return " | ".join(parts) if parts else ""
        except Exception:
            pass

    return ""


def borrar_modelo(nombre_modelo):
    """Borra el modelo ONNX seleccionado y su JSON companion del disco."""
    if not nombre_modelo or nombre_modelo == "No hay modelos subidos":
        return "Error: No hay modelo seleccionado.", "", "{}", ""

    rutas_borradas = []
    errores = []

    # Borrar ONNX
    onnx_path = os.path.join(UPLOAD_DIR, nombre_modelo)
    if os.path.exists(onnx_path):
        try:
            os.remove(onnx_path)
            rutas_borradas.append(f"ONNX: {nombre_modelo}")
        except Exception as e:
            errores.append(f"Error al borrar ONNX: {e}")

    # Borrar JSON companion
    json_path = _get_json_path(nombre_modelo)
    if os.path.exists(json_path):
        try:
            os.remove(json_path)
            rutas_borradas.append(f"JSON: {os.path.basename(json_path)}")
        except Exception as e:
            errores.append(f"Error al borrar JSON: {e}")

    if errores:
        status = f"⚠️ Parcialmente borrado: {', '.join(rutas_borradas)}. Errores: {'; '.join(errores)}"
    elif rutas_borradas:
        status = f"✅ Borrado: {', '.join(rutas_borradas)}"
    else:
        status = f"⚠️ No se encontraron archivos para {nombre_modelo}"

    return status, "*Selecciona un modelo para ver su información.*", "{}", ""

## **Cell 4 — GRADIO INTERFACE**

**Gradio interface with three tabs**

This cell builds the interactive web interface with Gradio 6, organized in three tabs.

*   The Inference tab allows loading models, configuring tile processing, and generating upscaled images with PNG and JPG download.
*   The Conversion tab allows transforming PyTorch models to ONNX with precision selection and optional OpenModelDB integration.
*   The Models tab shows the complete model card, allows generating information from OpenModelDB, editing personal comments, and deleting models from disk.

The interface launches with a temporary public link accessible from any browser.

In [ ]:
# ==========================================
# CELDA 4: Interfaz con 3 Tabs
# ==========================================
import os
import gradio as gr

with gr.Blocks(title="Escalador ONNX en Colab") as app:
    gr.Markdown("# 🚀 Escalador de Imágenes ONNX (Gradio 6 + UV)")

    with gr.Tabs():

        # ═══════════════════════════════════════════════════════
        # TAB 1 — INFERENCIA
        # ═══════════════════════════════════════════════════════
        with gr.Tab("🔍 Inferencia"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 1. Gestión de Modelos")
                    file_model_input = gr.File(label="Subir Modelo (.onnx)", file_types=[".onnx"])
                    chk_sobrescribir_subida = gr.Checkbox(
                        label="Sobrescribir si ya existe", value=False
                    )
                    btn_upload_model = gr.Button("Cargar Modelo al Disco", variant="secondary")
                    model_status = gr.Textbox(label="Estado del Modelo", interactive=False)

                    gr.Markdown("---")
                    gr.Markdown("### 2. Configuración del Modelo")
                    dropdown_modelos = gr.Dropdown(
                        choices=listar_modelos(),
                        value=listar_modelos()[0] if listar_modelos() else None,
                        label="Seleccionar Modelo Disponible"
                    )
                    btn_refresh_models = gr.Button("🔄 Actualizar lista", size="sm")
                    txt_info_modelo_basico = gr.Textbox(
                        label="Info del Modelo", interactive=False, value=""
                    )

                    gr.Markdown("---")
                    gr.Markdown("### 3. Ajustes de Tiling")
                    chk_use_tiling = gr.Checkbox(
                        label="Activar Tiling (Procesamiento por Parches)", value=True
                    )
                    txt_tile_sugerido = gr.Textbox(
                        label="Recomendación de Tiling",
                        interactive=False,
                        value="Carga una imagen para ver la recomendación"
                    )
                    with gr.Group():
                        radio_tile_size = gr.Radio(
                            choices=[256, 512, 768, 1024],
                            value=512,
                            label="Tamaño de Tile (Píxeles)"
                        )
                        radio_tile_pad = gr.Radio(
                            choices=[16, 32, 64, 128],
                            value=32,
                            label="Superposición / Overlap (Píxeles)"
                        )

                    gr.Markdown("---")
                    gr.Markdown("### 4. Escalado")
                    chk_doble_escala = gr.Checkbox(
                        label="Doble escalado (2 pases)",
                        value=False,
                        info="Ejecuta el modelo dos veces. Ej: 2x→4x, 4x→16x"
                    )

                    btn_run = gr.Button("⚡ Ejecutar Inferencia", variant="primary")

                with gr.Column(scale=2):
                    with gr.Row():
                        image_input = gr.Image(label="Imagen de Origen", type="filepath")
                        image_output = gr.Image(
                            label="Resultado (Vista Ligera Previa)",
                            type="filepath", format="png"
                        )
                    gr.Markdown("### 📥 Descargar Resultados")
                    with gr.Row():
                        btn_download_png = gr.DownloadButton("💾 Descargar PNG", variant="secondary")
                        btn_download_jpg = gr.DownloadButton("💾 Descargar JPG", variant="secondary")
                    gr.Markdown("### 📊 Información y Metadatos")
                    with gr.Row():
                        txt_dim_origen = gr.Textbox(label="Dimensiones Origen", interactive=False)
                        txt_peso_origen = gr.Textbox(label="Peso Origen", interactive=False)
                        txt_meta_salida = gr.Textbox(
                            label="Resultado (Dimensiones | Peso)", interactive=False
                        )
                    txt_exec_status = gr.Textbox(
                        label="Estado de Ejecución y Contador de Tiles",
                        interactive=False, value="Esperando inicio..."
                    )

        # ═══════════════════════════════════════════════════════
        # TAB 2 — CONVERSIÓN
        # ═══════════════════════════════════════════════════════
        with gr.Tab("🔄 Conversión"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 1. Modelo PyTorch")
                    file_pytorch = gr.File(
                        label="Subir modelo (.pth / .safetensors / .ckpt / .pt)",
                        file_types=[".pth", ".safetensors", ".ckpt", ".pt"],
                    )

                    gr.Markdown("---")
                    gr.Markdown("### 2. OpenModelDB (Opcional)")
                    txt_openmodeldb_url = gr.Textbox(
                        label="URL de OpenModelDB",
                        placeholder="https://openmodeldb.info/models/4x-AnimeSharp",
                    )

                    gr.Markdown("---")
                    gr.Markdown("### 3. Precisión de Salida")
                    radio_precision = gr.Radio(
                        choices=["fp32", "fp16"],
                        value="fp32",
                        label="Precisión (32-bit o 16-bit)",
                    )

                    gr.Markdown("---")
                    gr.Markdown("### 4. Conflicto de nombres")
                    chk_sobrescribir_conversion = gr.Checkbox(
                        label="Sobrescribir ONNX existente",
                        value=False,
                        info="Si está desmarcado, se renombrará automáticamente (ej: modelo_v2.onnx)"
                    )

                    btn_convert = gr.Button("🔄 Convertir a ONNX", variant="primary")
                    txt_convert_status = gr.Textbox(
                        label="Estado de Conversión", interactive=False, lines=4
                    )

                with gr.Column(scale=2):
                    txt_convert_info = gr.Textbox(
                        label="Información del Modelo Detectado",
                        interactive=False, lines=15,
                    )

        # ═══════════════════════════════════════════════════════
        # TAB 3 — MODELOS
        # ═══════════════════════════════════════════════════════
        with gr.Tab("📋 Modelos"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### Seleccionar Modelo")
                    dropdown_modelos_info = gr.Dropdown(
                        choices=listar_modelos(),
                        value=listar_modelos()[0] if listar_modelos() else None,
                        label="Modelos en disco",
                    )
                    btn_refresh_info = gr.Button("🔄 Actualizar lista", size="sm")

                    gr.Markdown("---")
                    gr.Markdown("### Generar Ficha desde OpenModelDB")
                    txt_url_modelos = gr.Textbox(
                        label="URL de OpenModelDB",
                        placeholder="https://openmodeldb.info/models/...",
                    )
                    btn_generar_json = gr.Button("📥 Generar Ficha", variant="secondary")

                    gr.Markdown("---")
                    gr.Markdown("### Borrar Modelo")
                    chk_confirmar_borrado = gr.Checkbox(
                        label="Confirmar borrado", value=False,
                        info="Marcar para habilitar el botón de borrado"
                    )
                    btn_borrar_modelo = gr.Button(
                        "🗑️ Borrar Modelo", variant="stop", interactive=False
                    )

                    txt_modelos_status = gr.Textbox(label="Estado", interactive=False)

                with gr.Column(scale=2):
                    md_modelos_info = gr.Markdown(
                        value="*Selecciona un modelo para ver su información.*"
                    )
                    gr.Markdown("### 💬 Comentarios Personales")
                    txt_modelos_comentarios = gr.Textbox(
                        label="Tus comentarios sobre este modelo",
                        lines=4, interactive=True,
                        placeholder="Escribe aquí tus notas, observaciones o valoraciones...",
                    )
                    btn_guardar_comentarios = gr.Button("💾 Guardar Comentarios", variant="primary")

                    with gr.Accordion("🔧 Ver JSON raw", open=False):
                        txt_modelos_json = gr.Textbox(
                            label="JSON", interactive=False, lines=15,
                        )

    # ═══════════════════════════════════════════════════════════
    # EVENTOS — INFERENCIA
    # ═══════════════════════════════════════════════════════════

    btn_upload_model.click(
        fn=guardar_modelo,
        inputs=[file_model_input, chk_sobrescribir_subida],
        outputs=[model_status, dropdown_modelos],
    ).then(
        fn=lambda: gr.Dropdown(choices=listar_modelos()),
        outputs=[dropdown_modelos_info],
    )

    btn_refresh_models.click(
        fn=lambda: gr.Dropdown(choices=listar_modelos()),
        inputs=None,
        outputs=[dropdown_modelos],
    )

    image_input.change(
        fn=obtener_metadatos_y_sugerencia,
        inputs=[image_input],
        outputs=[txt_dim_origen, txt_peso_origen, txt_tile_sugerido],
    )

    def ejecucion_completa(nombre_modelo, ruta_imagen, usar_tiling, tile_size, tile_pad, doble_escala):
        for res in procesar_imagen_onnx(
            nombre_modelo, ruta_imagen, usar_tiling, tile_size, tile_pad, doble_escala
        ):
            yield res

    btn_run.click(
        fn=ejecucion_completa,
        inputs=[
            dropdown_modelos, image_input, chk_use_tiling,
            radio_tile_size, radio_tile_pad, chk_doble_escala,
        ],
        outputs=[
            image_output, btn_download_png, btn_download_jpg,
            txt_exec_status, txt_dim_origen, txt_peso_origen, txt_meta_salida,
        ],
    )

    dropdown_modelos.change(
        fn=leer_info_basica_modelo,
        inputs=[dropdown_modelos],
        outputs=[txt_info_modelo_basico],
    )

    # ═══════════════════════════════════════════════════════════
    # EVENTOS — CONVERSIÓN
    # ═══════════════════════════════════════════════════════════

    def ejecucion_conversion(archivo, url, precision, sobrescribir):
        for res in convertir_pytorch_a_onnx(archivo, url, precision, sobrescribir):
            yield res

    btn_convert.click(
        fn=ejecucion_conversion,
        inputs=[file_pytorch, txt_openmodeldb_url, radio_precision, chk_sobrescribir_conversion],
        outputs=[txt_convert_status, txt_convert_info],
    ).then(
        fn=lambda: (
            gr.Dropdown(choices=listar_modelos()),
            gr.Dropdown(choices=listar_modelos()),
        ),
        outputs=[dropdown_modelos, dropdown_modelos_info],
    )

    # ═══════════════════════════════════════════════════════════
    # EVENTOS — MODELOS
    # ═══════════════════════════════════════════════════════════

    dropdown_modelos_info.change(
        fn=cargar_ficha_modelo,
        inputs=[dropdown_modelos_info],
        outputs=[md_modelos_info, txt_modelos_json, txt_modelos_comentarios],
    )

    btn_guardar_comentarios.click(
        fn=guardar_comentarios,
        inputs=[dropdown_modelos_info, txt_modelos_comentarios],
        outputs=[txt_modelos_status],
    )

    btn_generar_json.click(
        fn=generar_json_desde_url,
        inputs=[dropdown_modelos_info, txt_url_modelos],
        outputs=[txt_modelos_status, md_modelos_info, txt_modelos_json, txt_modelos_comentarios],
    )

    btn_refresh_info.click(
        fn=lambda: gr.Dropdown(choices=listar_modelos()),
        inputs=None,
        outputs=[dropdown_modelos_info],
    )

    # Habilitar/deshabilitar botón de borrado según checkbox
    chk_confirmar_borrado.change(
        fn=lambda checked: gr.Button(interactive=checked),
        inputs=[chk_confirmar_borrado],
        outputs=[btn_borrar_modelo],
    )

    btn_borrar_modelo.click(
        fn=borrar_modelo,
        inputs=[dropdown_modelos_info],
        outputs=[txt_modelos_status, md_modelos_info, txt_modelos_json, txt_modelos_comentarios],
    ).then(
        fn=lambda: (
            gr.Dropdown(choices=listar_modelos(), value=listar_modelos()[0] if listar_modelos() else None),
            gr.Dropdown(choices=listar_modelos(), value=listar_modelos()[0] if listar_modelos() else None),
        ),
        outputs=[dropdown_modelos_info, dropdown_modelos],
    )

    # Cargar ficha del modelo al iniciar la página
    app.load(
        fn=cargar_ficha_modelo,
        inputs=[dropdown_modelos_info],
        outputs=[md_modelos_info, txt_modelos_json, txt_modelos_comentarios],
    )

app.launch(share=True, debug=True)